In [ ]:
# import os
# os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

In [ ]:
rag_instruct_dataset_path = "./hf-dataset/RAG-Instruct"
pana_dataset_path = "./hf-dataset/panasonic_qa_v0"
model_path = "./models/Qwen3-4B"
tokenizer_path = "./models/Qwen3-4B"
gallery_path = "./gallery"
device = "cuda"
batch_train = 8
batch_eval = 8
grad_accumulation = 8
max_seq_len = 8196

In [3]:
from datasets import load_from_disk, concatenate_datasets, DatasetDict, load_dataset
instruct_ds = load_dataset(rag_instruct_dataset_path)

ds = load_from_disk(pana_dataset_path)
def normalize(example):
    # answer
    if example["answer"] is None:
        example["answer"] = []
    elif isinstance(example["answer"], str):
        example["answer"] = [example["answer"]]

    # documents
    if example["documents"] is None:
        example["documents"] = []
    elif isinstance(example["documents"], str):
        example["documents"] = [example["documents"]]

    return example

for split in ['r0', 'r1', 'r2', 'r3', 'r4']:
    ds[split] = ds[split].map(normalize)

instruct_ds['train'] = instruct_ds['train'].map(normalize)

all_datasets = []
for split in ['r0', 'r1', 'r2', 'r3', 'r4']:
    all_datasets.append(ds[split])

all_datasets.append(instruct_ds['train'])

combined_ds = concatenate_datasets(all_datasets)
combined_ds
split_ds = combined_ds.train_test_split(test_size=0.05, seed=42)

dataset = DatasetDict({
    'train': split_ds['train'],
    'test': split_ds['test']
})
dataset

/home/parsa/.conda/envs/unsloth/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'documents'],
        num_rows: 60537
    })
    test: Dataset({
        features: ['question', 'answer', 'documents'],
        num_rows: 3187
    })
})

In [4]:
from unsloth import FastLanguageModel
import torch
dtype = torch.float32
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_len,
    dtype = dtype,
    device_map = "balanced",
    load_in_4bit = load_in_4bit,
    full_finetuning = True,
    # float32_mixed_precision = True
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 2. Max memory: 31.348 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.54s/it]


./models/Llama-3.2-3B does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


In [6]:
from unsloth.chat_templates import CHAT_TEMPLATES
print(list(CHAT_TEMPLATES.keys()))

['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'starling', 'yi-chat']


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "Qwen3"
)

def normalize_documents(documents):
    flat_docs = []
    for doc in documents:
        if isinstance(doc, list):
            flat_docs.append(" ".join(map(str, doc)))
        elif isinstance(doc, dict):
            flat_docs.append(" ".join(f"{k}: {v}" for k, v in doc.items()))
        else:
            flat_docs.append(str(doc))
    return flat_docs

def formatting_prompts_func(example):
    question = example["question"]
    documents = normalize_documents(example["documents"])
    answer = str(example["answer"])
    prompt = """
        Based on relevat document answer this question.
        relevant document: {}
        question: {}
    """
    input = prompt.format("\n".join(documents), question)

    texts = tokenizer.apply_chat_template(
        [
            {"role":"user", "content": input},
            {"role":"assistant", "content": answer}
        ],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text" : texts, }

In [8]:
# Map the dataset on function
# Here may face the error ValueError: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute
# This error is because this model is not instruct so in tokenizer_config.json there is no "chat_template". So we can find another chat template from instruct version and add it to this model.
dataset['train'] = dataset['train'].map(formatting_prompts_func)
dataset['test'] = dataset['test'].map(formatting_prompts_func)

Map: 100%|██████████| 3187/3187 [00:00<00:00, 11269.00 examples/s]


In [ ]:
dataset['train'][100]['text']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n\n        Based on relevat document answer this question.\n        relevant document: with Israel, the network resisted Israeli attempts to jam it, and Hezbollah maintained communications throughout the conflict. Hezbollah fighters mostly communicated using codewords on low-tech walkie-talkies, while command posts and bunkers were linked by the group\'s fiber optic network. Hezbollah relies heavily on cell phones to conduct its operations, both using existing Lebanese carriers and operating its own cellular networks. Limited numbers of high-ranking and critical personnel have satellite phones as a redundant measure. Hezbollah\'s communications network has greatly increased since 2006, and fiber optic cables links the homes of top commanders to bunkers and headquarters. Normal personnel have access\nover 

: 

In [8]:
# # TODO: Complete the compute metrics based on evaluation
# # Check : https://huggingface.co/docs/evaluate/index
# import numpy as np
# import evaluate
# import random

# exact_match_metric = evaluate.load("exact_match")

# def compute_metrics(eval_preds):
#     logits, labels = eval_preds

#     # Move tensors to CPU and get argmax safely
#     preds = tokenizer.batch_decode(
#         np.where(labels != tokenizer.pad_token_id, logits.argmax(-1), tokenizer.pad_token_id),
#         skip_special_tokens=True
#     )
#     refs = tokenizer.batch_decode(labels, skip_special_tokens=True)

#     # Strip whitespace
#     pred_str = [p.strip() for p in preds]
#     label_str = [l.strip() for l in refs]

#     # Compute exact match
#     exact_match = exact_match_metric.compute(
#         predictions=pred_str,
#         references=label_str
#     )

#     # Pick 2 random samples
#     indices = random.sample(range(len(pred_str)), k=min(2, len(pred_str)))
#     examples_shown = []
#     for i in indices:
#         print(f"pred: {pred_str[i]}")
#         print(f"ref : {label_str[i]}")
#         print("-"*20)
#         examples_shown.append({"pred": pred_str[i], "ref": label_str[i]})

#     # Return metrics
#     return {
#         "exact_match": exact_match["exact_match"],
#         "examples_shown": examples_shown
#     }


In [ ]:
from trl import SFTConfig
num_of_reports = 32

model_name = model_path.split('/')[-1]
dataset_name = pana_dataset_path.split('/')[-1]
second_dataset_name = rag_instruct_dataset_path.split('/')[-1]

SAVE_EVAL_LOG_STEPS = 1 / num_of_reports

ft_model_id = f'{gallery_path}/{model_name}-ft-{dataset_name}-{second_dataset_name}-2x8-sft'

args = SFTConfig(
    ft_model_id,
    run_name = ft_model_id,
    dataset_text_field = "text",

    per_device_train_batch_size=batch_train,
    per_device_eval_batch_size=batch_eval,
    gradient_accumulation_steps=grad_accumulation,

    num_train_epochs=1,
    learning_rate=5e-5,

    eval_strategy='steps',
    save_strategy='steps',
    logging_strategy='steps',

    save_steps=SAVE_EVAL_LOG_STEPS,
    eval_steps=SAVE_EVAL_LOG_STEPS,
    logging_steps=SAVE_EVAL_LOG_STEPS,

    save_total_limit = 2,
    
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    report_to='tensorboard',
    seed=42,
    eval_on_start = True,
    torch_compile=False,
    torch_compile_backend=None,
)

In [10]:
from trl import SFTTrainer


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    # compute_metrics = compute_metrics,
    args = args,
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [12]:
stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 60,537 | Num Epochs = 1 | Total steps = 7,568
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,022,468,096 of 4,022,468,096 (100.00% trained)
Unsloth: Not an error, but Qwen3Model does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.model.save_pretrained(f"{ft_model_id}")
trainer.tokenizer.save_pretrained(f"{ft_model_id}")

: 